In [12]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer ,PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB  
from hmmlearn.hmm import GaussianHMM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model

# from sktime.detection.hmm_learn import GaussianHMM 

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,ConfusionMatrixDisplay

from sklearn.decomposition import TruncatedSVD #instead of PCA
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [2]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
stop_words=set(stopwords.words('english'))

# Load the data + class Weights

In [3]:
data = joblib.load('news_data.pkl')
x_train = data['X_train']
y_train = data['y_train']
x_test = data['X_test']
y_test = data['y_test']

# data_balance = joblib.load('news_data_resampled.pkl')
# x_train_ran_res = data_balance['X_train']
# y_train_ran_res = data_balance['y_train']
# x_test_ran_res = data_balance['X_test']
# y_test_ran_res = data_balance['y_test']

# data_balance_rosrus=joblib.load('news_data_bal_ros_rus.pkl') #done
# x_train_bal = data_balance_rosrus['X_train']
# y_train_bal = data_balance_rosrus['y_train']
# x_test_bal = data_balance_rosrus['X_test']
# y_test_bal = data_balance_rosrus['y_test']

In [5]:
data_undersampled=joblib.load('news_data_undersampled.pkl') # in progress
x_train_undersampled=data_undersampled['X_train']
y_train_undersampled=data_undersampled['y_train']
x_test_undersampled=data_undersampled['X_test']
y_test_undersampled=data_undersampled['y_test']

In [6]:
class_weights_dict=joblib.load('classWeightsDic')

In [7]:
results_Bow_rosrus=joblib.load("results_Bow_rosrus")

# Initial Stem & Lemma

In [4]:
stemmer=PorterStemmer()
lemmatizer=WordNetLemmatizer()

In [5]:
def stem(text,remove_stopwords=False):
    tokens=word_tokenize(text.lower())
    if remove_stopwords:
        tokens= [t for t in tokens if t not in stop_words]
    results=[stemmer.stem(t) for t in tokens]
    return results

In [6]:
def lemma(text,remove_stopwords=False):
    tokens=word_tokenize(text.lower())
    if remove_stopwords:
        tokens= [t for t in tokens if t not in stop_words]
    results=[lemmatizer.lemmatize(t) for t in tokens]
    return results

In [7]:
num_classes = len(set(y_train))  
num_classes

10

# Original Data

In [12]:
len(x_train)

167616

## BoW feature Extraction

## count vectorizer

In [13]:
vectorizer_stem = CountVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=True))
X_train_stem = vectorizer_stem.fit_transform(x_train)
x_test_stem = vectorizer_stem.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [8]:
vectorizer_stem_stopKept = CountVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=False))
x_train_stem_stopkept = vectorizer_stem_stopKept.fit_transform(x_train)
x_test_stem_stopkept = vectorizer_stem_stopKept.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [15]:
vectorizer_lemma = CountVectorizer(tokenizer=lambda x:lemma(x, remove_stopwords=True))
x_train_lemma = vectorizer_lemma.fit_transform(x_train)
x_test_lemma = vectorizer_lemma.transform(x_test)

In [16]:
vectorizer_lemma_stopkept = CountVectorizer(tokenizer=lambda x: lemma(x, remove_stopwords=False))
x_train_lemma_stopkept = vectorizer_lemma_stopkept.fit_transform(x_train)
x_test_lemma_stopkept = vectorizer_lemma_stopkept.transform(x_test)

## TF-IDF vectorizer

In [17]:
vectorizer_stem_tfidf = TfidfVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=True))
x_train_stem_tfidf = vectorizer_stem_tfidf.fit_transform(x_train)
x_test_stem_tfidf = vectorizer_stem_tfidf.transform(x_test)

In [18]:
vectorizer_stem_stopkept_tfidf = TfidfVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=True))
X_train_stem_stopkept_tfidf = vectorizer_stem_stopkept_tfidf.fit_transform(x_train)
x_test_stem_stopkept_tfidf = vectorizer_stem_stopkept_tfidf.transform(x_test)

In [19]:
vectorizer_lemma_tfidf = TfidfVectorizer(tokenizer=lambda x: lemma(x, remove_stopwords=True))
X_train_lemma_tfidf = vectorizer_lemma_tfidf.fit_transform(x_train)
x_test_lemma_tfidf = vectorizer_lemma_tfidf.transform(x_test)

In [20]:
vectorizer_lemma_stopkept_tfidf = TfidfVectorizer(tokenizer=lambda x: lemma(x, remove_stopwords=True))
X_train_lemma_stopkept_tfidf = vectorizer_lemma_stopkept_tfidf.fit_transform(x_train)
x_test_lemma_stopkept_tfidf = vectorizer_lemma_stopkept_tfidf.transform(x_test)

## Models

In [ ]:
# results_Bow={}
# results_Bow=joblib.load('results_Bow')

In [21]:
results_Bow_orginal=joblib.load('results_BoW_original')
results_Bow_orginal

{'SVC_BoW stem+StopWords kept': 0.6616155590025057,
 'MultinomialNB_BoW stem+StopWords kept': 0.6672234816847632,
 'SVC_BoW Lemma+StopWords kept': 0.6605894284691565,
 'MultinomialNB_BoW Lemma+StopWords kept': 0.6684643837250924,
 'SVC_BoW stem+StopWords removed': 0.6614007874955256,
 'MultinomialNB_BoW stem+StopWords removed': 0.6731177663763274,
 'SVC_BoW lemma+StopWords removed': 0.6615439685001789,
 'MultinomialNB_BoW lemma+StopWords removed': 0.6737859443980432,
 'HMM stem+StopWords removed': 0.04331225390764825,
 'HMM stem+StopWords kept': 0.1699081255220141,
 'HMM lemma+StopWords kept': 0.06261782603507934,
 'HMM lemma+StopWords removed': 0.06369168356997972,
 'MLP stem+stopwords removed': 0.5851569025175993,
 'MLP stem+stopwords kept': 0.546498031261186,
 'MLP lemma+stopwords removed': 0.5852523565207016,
 'MLP lemma+stopwords kept': 0.5464503042596349,
 '(tf-idf) MultinomialNB_BoW stem+StopWords removed': 0.6731177663763274,
 '(tf-idf) SVC_BoW stem+StopWords removed': 0.661400

In [158]:
models_Bow={
    "SVC_BoW": SVC(class_weight=class_weights_dict),
    "MultinomialNB_BoW": MultinomialNB(),
    # "MLP" :MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42),
    }

## with countvectorizer

### Stem + stop words kept

In [11]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)


SVCnocw=SVC()
SVCnocw.fit(x_train_stem_stopkept, y_train)
y_pred = SVCnocw.predict(x_test_stem_stopkept) #kept
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for SVC(): accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"cat_HMMXBOW_confusion_matrix.png", dpi=300)
plt.close()

Results for SVC(): accuracy=0.7009664717814104
              precision    recall  f1-score   support

           1       0.81      0.71      0.76      7120
           2       0.73      0.43      0.54      3589
           3       0.76      0.55      0.64      3473
           4       0.83      0.55      0.66      1980
           5       0.88      0.62      0.73      1963
           6       0.79      0.52      0.63      1269
           7       0.76      0.49      0.59      1268
           8       0.79      0.14      0.23      1198
           9       0.74      0.39      0.51      1016
          10       0.64      0.88      0.74     19029

    accuracy                           0.70     41905
   macro avg       0.77      0.53      0.60     41905
weighted avg       0.72      0.70      0.69     41905

--------------------------------------------------


AttributeError: module 'matplotlib' has no attribute 'subplots'

In [13]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"cat_HMMXBOW_confusion_matrix.png", dpi=300)
plt.close()

In [ ]:
for model_name, model in models_Bow.items():
    model.fit(x_train_stem_stopkept, y_train)
    y_pred = model.predict(x_test_stem_stopkept) #kept
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_Bow_orginal[model_name+" stem+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.6616155590025057
              precision    recall  f1-score   support

           1       0.73      0.78      0.75      7120
           2       0.44      0.79      0.57      3589
           3       0.56      0.77      0.65      3473
           4       0.66      0.74      0.70      1980
           5       0.69      0.78      0.73      1963
           6       0.73      0.60      0.66      1269
           7       0.60      0.73      0.66      1268
           8       0.41      0.52      0.46      1198
           9       0.63      0.61      0.62      1016
          10       0.79      0.56      0.66     19029

    accuracy                           0.66     41905
   macro avg       0.62      0.69      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6672234816847632
              precision    recall  f1-score   support

           1       0.70    

TypeError: Sparse data was passed, but dense data is required. Use '.toarray()' to convert to a dense numpy array.

In [18]:
len(x_train)

167616

### lemma + stopwords kept

In [ ]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_lemma_stopkept, y_train)
    y_pred = model.predict(x_test_lemma_stopkept) #kept
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results_Bow_orginal[model_name+" Lemma+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.6605894284691565
              precision    recall  f1-score   support

           1       0.73      0.77      0.75      7120
           2       0.44      0.79      0.56      3589
           3       0.56      0.77      0.64      3473
           4       0.67      0.74      0.70      1980
           5       0.70      0.77      0.74      1963
           6       0.74      0.60      0.66      1269
           7       0.60      0.73      0.66      1268
           8       0.43      0.51      0.46      1198
           9       0.65      0.61      0.63      1016
          10       0.79      0.57      0.66     19029

    accuracy                           0.66     41905
   macro avg       0.63      0.69      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6684643837250924
              precision    recall  f1-score   support

           1       0.71    

### Stem + remove stop words

In [ ]:
for model_name, model in models_Bow.items():    
    model.fit(X_train_stem, y_train)
    y_pred = model.predict(x_test_stem) #removed
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results_Bow_orginal[model_name+" stem+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.6614007874955256
              precision    recall  f1-score   support

           1       0.73      0.79      0.76      7120
           2       0.44      0.79      0.57      3589
           3       0.56      0.77      0.65      3473
           4       0.66      0.75      0.71      1980
           5       0.68      0.79      0.73      1963
           6       0.71      0.62      0.66      1269
           7       0.58      0.75      0.66      1268
           8       0.41      0.55      0.47      1198
           9       0.63      0.63      0.63      1016
          10       0.80      0.55      0.65     19029

    accuracy                           0.66     41905
   macro avg       0.62      0.70      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6731177663763274
              precision    recall  f1-score   support

           1       0.71    

### Lemma + remove stop words

In [ ]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_lemma, y_train)
    y_pred = model.predict(x_test_lemma) #removed
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results_Bow_orginal[model_name+" lemma+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.6615439685001789
              precision    recall  f1-score   support

           1       0.73      0.78      0.76      7120
           2       0.43      0.80      0.56      3589
           3       0.57      0.76      0.65      3473
           4       0.68      0.75      0.71      1980
           5       0.69      0.79      0.73      1963
           6       0.73      0.62      0.67      1269
           7       0.59      0.74      0.66      1268
           8       0.42      0.53      0.47      1198
           9       0.64      0.63      0.63      1016
          10       0.79      0.56      0.66     19029

    accuracy                           0.66     41905
   macro avg       0.63      0.70      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6737859443980432
              precision    recall  f1-score   support

           1       0.71    

### HMM

In [ ]:
hmm_stem_orig = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)
hmm_stem_stopkept_orig = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)
hmm_lemma_orig = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)
hmm_lemma_stopkept_orig = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)

#### dimension reduction (66774,50)

In [76]:
stem_svd_orgin=TruncatedSVD(n_components=50)
x_train_stem_orgind_svd=stem_svd_orgin.fit_transform(X_train_stem)
x_test_stem_orgind_svd=stem_svd_orgin.transform(x_test_stem)

In [77]:
stem_stopkept_svd_orgin=TruncatedSVD(n_components=50)
x_train_stem_stopkept_orgind_svd=stem_stopkept_svd_orgin.fit_transform(x_train_stem_stopkept)
x_test_stem_stopkept_orgind_svd=stem_stopkept_svd_orgin.transform(x_test_stem_stopkept)

In [78]:
lemma_svd_orgin=TruncatedSVD(n_components=50)
x_train_lemma_orgind_svd=lemma_svd_orgin.fit_transform(x_train_lemma)
x_test_lemma_orgind_svd=lemma_svd_orgin.transform(x_test_lemma)

In [79]:
lemma_stopkept_svd_orgin=TruncatedSVD(n_components=50)
x_train_lemma_stopkept_orgind_svd=lemma_stopkept_svd_orgin.fit_transform(x_train_lemma_stopkept)
x_test_lemma_stopkept_orgind_svd=lemma_stopkept_svd_orgin.transform(x_test_lemma_stopkept)

#### apply HMM

In [80]:
lengths_train = [1] * x_train_stem_orgind_svd.shape[0]
lengths_test = [1] * x_test_stem_orgind_svd.shape[0]

##### stem

In [98]:
hmm_stem_orig.fit(x_train_stem_orgind_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [99]:
y_pred_stem_orgin=hmm_stem_orig.predict(x_train_stem_orgind_svd)
accuracy_hmm_stem_orgin = accuracy_score(y_train, y_pred_stem_orgin)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_orgin}")

training HMM BoW Accuracy with lengths: 0.04074193394425353


In [87]:
y_pred_stem_orgin=hmm_stem_orig.predict(x_test_stem_orgind_svd,lengths=lengths_test)
accuracy_hmm_stem_orgin = accuracy_score(y_test, y_pred_stem_orgin)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_orgin}")

testing HMM BoW Accuracy with lengths: 0.030282782484190432


In [88]:
y_pred_stem_orgin=hmm_stem_orig.predict(x_test_stem_orgind_svd)
accuracy_hmm_stem_orgin = accuracy_score(y_test, y_pred_stem_orgin)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_orgin}")

testing HMM BoW Accuracy: 0.04331225390764825


In [101]:
results_Bow_orginal['HMM stem+StopWords removed']=0.04331225390764825

##### stem + stopwords kept

In [89]:
hmm_stem_stopkept_orig.fit(x_train_stem_stopkept_orgind_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [90]:
y_pred_stem_stopkept_orgin=hmm_stem_stopkept_orig.predict(x_train_stem_stopkept_orgind_svd,lengths=lengths_train)
accuracy_hmm_stem_stopkept_orgin = accuracy_score(y_train, y_pred_stem_stopkept_orgin)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_stopkept_orgin}")

training HMM BoW Accuracy with lengths: 0.16991814623902252


In [91]:
y_pred_stem_stopkept_orgin=hmm_stem_stopkept_orig.predict(x_test_stem_stopkept_orgind_svd,lengths=lengths_test)
accuracy_hmm_stem_stopkept_orgin = accuracy_score(y_test, y_pred_stem_stopkept_orgin)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_stopkept_orgin}")

testing HMM BoW Accuracy with lengths: 0.1699081255220141


In [92]:
y_pred_stem_stopkept_orgin=hmm_stem_stopkept_orig.predict(x_test_stem_stopkept_orgind_svd)
accuracy_hmm_stem_stopkept_orgin = accuracy_score(y_test, y_pred_stem_stopkept_orgin)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_stopkept_orgin}")

testing HMM BoW Accuracy: 0.047774728552678676


In [105]:
results_Bow_orginal['HMM stem+StopWords kept']=0.1699081255220141

##### lemma

In [116]:
hmm_lemma_orig.fit(x_train_lemma_orgind_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [122]:
y_pred_lemma_orgin=hmm_lemma_orig.predict(x_train_lemma_orgind_svd,)
accuracy_hmm_lemma_orgin = accuracy_score(y_train, y_pred_lemma_orgin)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_lemma_orgin}")

training HMM BoW Accuracy : 0.06319205803741887


In [120]:
y_pred_lemma_orgin=hmm_lemma_orig.predict(x_test_lemma_orgind_svd,lengths=lengths_test)
accuracy_hmm_lemma_orgin = accuracy_score(y_test, y_pred_lemma_orgin)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_orgin}")

testing HMM BoW Accuracy with lengths: 0.04684405202243169


In [121]:
y_pred_lemma_orgin=hmm_lemma_orig.predict(x_test_lemma_orgind_svd)
accuracy_hmm_lemma_orgin = accuracy_score(y_test, y_pred_lemma_orgin)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_lemma_orgin}")

testing HMM BoW Accuracy: 0.06369168356997972


In [123]:
results_Bow_orginal['HMM lemma+StopWords removed']=0.06369168356997972

##### lemma + stopwords kept

In [109]:
hmm_lemma_stopkept_orig.fit(x_train_lemma_stopkept_orgind_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [113]:
y_pred_lemma_stopkept_orgin=hmm_lemma_stopkept_orig.predict(x_train_lemma_stopkept_orgind_svd)
accuracy_hmm_lemma_stopkept_orgin = accuracy_score(y_train, y_pred_lemma_stopkept_orgin)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_stopkept_orgin}")

training HMM BoW Accuracy with lengths: 0.0640750286368843


In [111]:
y_pred_lemma_stopkept_orgin=hmm_lemma_stopkept_orig.predict(x_test_lemma_stopkept_orgind_svd,lengths=lengths_test)
accuracy_hmm_lemma_stopkept_orgin = accuracy_score(y_test, y_pred_lemma_stopkept_orgin)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_stopkept_orgin}")

testing HMM BoW Accuracy with lengths: 0.04724973153561628


In [112]:
y_pred_lemma_stopkept_orgin=hmm_lemma_stopkept_orig.predict(x_test_lemma_stopkept_orgind_svd)
accuracy_hmm_lemma_stopkept_orgin = accuracy_score(y_test, y_pred_lemma_stopkept_orgin)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_lemma_stopkept_orgin}")

testing HMM BoW Accuracy: 0.06261782603507934


In [114]:
results_Bow_orginal['HMM lemma+StopWords kept']=0.06261782603507934

##### save HMM results

In [124]:
results_Bow_orginal

{'SVC_BoW stem+StopWords kept': 0.6616155590025057,
 'MultinomialNB_BoW stem+StopWords kept': 0.6672234816847632,
 'SVC_BoW Lemma+StopWords kept': 0.6605894284691565,
 'MultinomialNB_BoW Lemma+StopWords kept': 0.6684643837250924,
 'SVC_BoW stem+StopWords removed': 0.6614007874955256,
 'MultinomialNB_BoW stem+StopWords removed': 0.6731177663763274,
 'SVC_BoW lemma+StopWords removed': 0.6615439685001789,
 'MultinomialNB_BoW lemma+StopWords removed': 0.6737859443980432,
 'HMM stem+StopWords removed': 0.04331225390764825,
 'HMM stem+StopWords kept': 0.1699081255220141,
 'HMM lemma+StopWords kept': 0.06261782603507934,
 'HMM lemma+StopWords removed': 0.06369168356997972}

In [125]:
joblib.dump(results_Bow_orginal,'results_Bow_orginal')

['results_Bow_orginal']

### MLP With dimension reduction

In [126]:
x_train_stem_stopkept_orgind_svd.shape

(167616, 50)

In [127]:
orginal_stem_MLP=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
orginal_stem_stopkept_MLP=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
orginal_lemma_MLP=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
orginal_lemma_stopkept_MLP=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)

#### stem

In [129]:
orginal_stem_MLP.fit(x_train_stem_orgind_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [133]:
y_pred_train=orginal_stem_MLP.predict(x_train_stem_orgind_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.596040950744559
              precision    recall  f1-score   support

           1       0.70      0.65      0.67     28481
           2       0.57      0.27      0.37     14356
           3       0.51      0.35      0.41     13889
           4       0.64      0.30      0.41      7920
           5       0.71      0.54      0.62      7851
           6       0.73      0.18      0.29      5077
           7       0.57      0.33      0.42      5072
           8       0.87      0.06      0.11      4793
           9       0.53      0.02      0.03      4061
          10       0.57      0.83      0.68     76116

    accuracy                           0.60    167616
   macro avg       0.64      0.35      0.40    167616
weighted avg       0.61      0.60      0.56    167616



In [134]:
y_pred_test=orginal_stem_MLP.predict(x_test_stem_orgind_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.5851569025175993
              precision    recall  f1-score   support

           1       0.69      0.63      0.66      7120
           2       0.55      0.26      0.36      3589
           3       0.49      0.34      0.40      3473
           4       0.66      0.29      0.41      1980
           5       0.69      0.52      0.60      1963
           6       0.73      0.18      0.28      1269
           7       0.57      0.30      0.39      1268
           8       0.82      0.05      0.10      1198
           9       0.46      0.02      0.03      1016
          10       0.56      0.82      0.67     19029

    accuracy                           0.59     41905
   macro avg       0.62      0.34      0.39     41905
weighted avg       0.60      0.59      0.55     41905



In [136]:
results_Bow_orginal['MLP stem+stopwords removed']=0.5851569025175993

#### stem + stopwords kept

In [138]:
orginal_stem_stopkept_MLP.fit(x_train_stem_stopkept_orgind_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [139]:
y_pred_train=orginal_stem_stopkept_MLP.predict(x_train_stem_stopkept_orgind_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.5590755059182895
              precision    recall  f1-score   support

           1       0.62      0.64      0.63     28481
           2       0.53      0.16      0.25     14356
           3       0.53      0.21      0.30     13889
           4       0.64      0.13      0.21      7920
           5       0.68      0.47      0.56      7851
           6       0.58      0.01      0.02      5077
           7       0.56      0.20      0.29      5072
           8       0.83      0.05      0.10      4793
           9       0.55      0.00      0.01      4061
          10       0.54      0.84      0.66     76116

    accuracy                           0.56    167616
   macro avg       0.61      0.27      0.30    167616
weighted avg       0.57      0.56      0.50    167616



In [140]:
y_pred_test=orginal_stem_stopkept_MLP.predict(x_test_stem_stopkept_orgind_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.546498031261186
              precision    recall  f1-score   support

           1       0.61      0.62      0.62      7120
           2       0.47      0.16      0.24      3589
           3       0.49      0.20      0.28      3473
           4       0.64      0.13      0.22      1980
           5       0.66      0.45      0.53      1963
           6       0.50      0.01      0.02      1269
           7       0.50      0.16      0.24      1268
           8       0.69      0.05      0.09      1198
           9       0.38      0.00      0.01      1016
          10       0.53      0.83      0.65     19029

    accuracy                           0.55     41905
   macro avg       0.55      0.26      0.29     41905
weighted avg       0.55      0.55      0.49     41905



In [141]:
results_Bow_orginal['MLP stem+stopwords kept']=0.546498031261186

#### lemma

In [142]:
orginal_lemma_MLP.fit(x_train_lemma_orgind_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [143]:
y_pred_train=orginal_lemma_MLP.predict(x_train_lemma_orgind_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.5976040473463153
              precision    recall  f1-score   support

           1       0.70      0.65      0.67     28481
           2       0.60      0.26      0.36     14356
           3       0.55      0.32      0.40     13889
           4       0.61      0.30      0.40      7920
           5       0.74      0.52      0.61      7851
           6       0.73      0.21      0.32      5077
           7       0.65      0.27      0.38      5072
           8       0.84      0.06      0.12      4793
           9       0.65      0.01      0.01      4061
          10       0.57      0.85      0.68     76116

    accuracy                           0.60    167616
   macro avg       0.66      0.34      0.40    167616
weighted avg       0.62      0.60      0.56    167616



In [144]:
y_pred_test=orginal_lemma_MLP.predict(x_test_lemma_orgind_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.5852523565207016
              precision    recall  f1-score   support

           1       0.69      0.63      0.66      7120
           2       0.56      0.24      0.34      3589
           3       0.52      0.30      0.38      3473
           4       0.61      0.28      0.39      1980
           5       0.72      0.50      0.59      1963
           6       0.72      0.20      0.31      1269
           7       0.65      0.24      0.35      1268
           8       0.76      0.06      0.11      1198
           9       0.56      0.00      0.01      1016
          10       0.56      0.84      0.67     19029

    accuracy                           0.59     41905
   macro avg       0.63      0.33      0.38     41905
weighted avg       0.60      0.59      0.55     41905



In [145]:
results_Bow_orginal['MLP lemma+stopwords removed']=0.5852523565207016

#### lemma + stopwords kept

In [146]:
orginal_lemma_stopkept_MLP.fit(x_train_lemma_stopkept_orgind_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [147]:
y_pred_train=orginal_lemma_stopkept_MLP.predict(x_train_lemma_stopkept_orgind_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.5601911512027491
              precision    recall  f1-score   support

           1       0.64      0.62      0.63     28481
           2       0.53      0.16      0.24     14356
           3       0.51      0.24      0.32     13889
           4       0.54      0.16      0.25      7920
           5       0.70      0.46      0.56      7851
           6       0.73      0.01      0.02      5077
           7       0.62      0.20      0.31      5072
           8       0.84      0.06      0.11      4793
           9       0.51      0.00      0.01      4061
          10       0.54      0.85      0.66     76116

    accuracy                           0.56    167616
   macro avg       0.62      0.28      0.31    167616
weighted avg       0.58      0.56      0.50    167616



In [148]:
y_pred_test=orginal_lemma_stopkept_MLP.predict(x_test_lemma_stopkept_orgind_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.5464503042596349
              precision    recall  f1-score   support

           1       0.63      0.61      0.62      7120
           2       0.45      0.14      0.22      3589
           3       0.47      0.23      0.31      3473
           4       0.52      0.16      0.24      1980
           5       0.67      0.44      0.53      1963
           6       0.48      0.01      0.02      1269
           7       0.53      0.16      0.25      1268
           8       0.77      0.05      0.09      1198
           9       0.50      0.00      0.01      1016
          10       0.53      0.83      0.65     19029

    accuracy                           0.55     41905
   macro avg       0.56      0.26      0.29     41905
weighted avg       0.55      0.55      0.49     41905



In [149]:
results_Bow_orginal['MLP lemma+stopwords kept']=0.5464503042596349

## with tfidfvectorizer

### stem

In [ ]:
for model_name, model in models_Bow.items():
    model.fit(x_train_stem_tfidf, y_train)
    y_pred = model.predict(x_test_stem_tfidf) #removed
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_Bow_orginal["(tf-idf) "+model_name+" stem+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.6614007874955256
              precision    recall  f1-score   support

           1       0.73      0.79      0.76      7120
           2       0.44      0.79      0.57      3589
           3       0.56      0.77      0.65      3473
           4       0.66      0.75      0.71      1980
           5       0.68      0.79      0.73      1963
           6       0.71      0.62      0.66      1269
           7       0.58      0.75      0.66      1268
           8       0.41      0.55      0.47      1198
           9       0.63      0.63      0.63      1016
          10       0.80      0.55      0.65     19029

    accuracy                           0.66     41905
   macro avg       0.62      0.70      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6731177663763274
              precision    recall  f1-score   support

           1       0.71    

### stem + stopwords kept

In [170]:
for model_name, model in models_Bow.items():
    model.fit(X_train_stem_stopkept_tfidf, y_train)
    y_pred = model.predict(x_test_stem_stopkept_tfidf) #kept
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_Bow_orginal["(tf-idf) "+model_name+" stem+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.6614007874955256
              precision    recall  f1-score   support

           1       0.73      0.79      0.76      7120
           2       0.44      0.79      0.57      3589
           3       0.56      0.77      0.65      3473
           4       0.66      0.75      0.71      1980
           5       0.68      0.79      0.73      1963
           6       0.71      0.62      0.66      1269
           7       0.58      0.75      0.66      1268
           8       0.41      0.55      0.47      1198
           9       0.63      0.63      0.63      1016
          10       0.80      0.55      0.65     19029

    accuracy                           0.66     41905
   macro avg       0.62      0.70      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6731177663763274
              precision    recall  f1-score   support

           1       0.71    

### lemma

In [172]:
for model_name, model in models_Bow.items():
    model.fit(X_train_lemma_tfidf, y_train)
    y_pred = model.predict(x_test_lemma_tfidf) #removed
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results_Bow_orginal["(tf-idf) "+model_name+" lemma+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.6615439685001789
              precision    recall  f1-score   support

           1       0.73      0.78      0.76      7120
           2       0.43      0.80      0.56      3589
           3       0.57      0.76      0.65      3473
           4       0.68      0.75      0.71      1980
           5       0.69      0.79      0.73      1963
           6       0.73      0.62      0.67      1269
           7       0.59      0.74      0.66      1268
           8       0.42      0.53      0.47      1198
           9       0.64      0.63      0.63      1016
          10       0.79      0.56      0.66     19029

    accuracy                           0.66     41905
   macro avg       0.63      0.70      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6737859443980432
              precision    recall  f1-score   support

           1       0.71    

### lemma + stopwords kept

In [173]:
for model_name, model in models_Bow.items():
    model.fit(X_train_lemma_stopkept_tfidf, y_train)
    y_pred = model.predict(x_test_lemma_stopkept_tfidf) #kept
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_Bow_orginal["(tf-idf) "+model_name+" lemma+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.6615439685001789
              precision    recall  f1-score   support

           1       0.73      0.78      0.76      7120
           2       0.43      0.80      0.56      3589
           3       0.57      0.76      0.65      3473
           4       0.68      0.75      0.71      1980
           5       0.69      0.79      0.73      1963
           6       0.73      0.62      0.67      1269
           7       0.59      0.74      0.66      1268
           8       0.42      0.53      0.47      1198
           9       0.64      0.63      0.63      1016
          10       0.79      0.56      0.66     19029

    accuracy                           0.66     41905
   macro avg       0.63      0.70      0.65     41905
weighted avg       0.70      0.66      0.66     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6737859443980432
              precision    recall  f1-score   support

           1       0.71    

### HMM

In [184]:
hmm_stem_original_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_stem_stopkept_original_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_original_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_stopkept_original_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,50)

In [ ]:
stem_svd_original_tfidf=TruncatedSVD(n_components=50)
x_train_stem_originald_tfidf_svd=stem_svd_original_tfidf.fit_transform(x_train_stem_tfidf)
x_test_stem_originald_tfidf_svd=stem_svd_original_tfidf.transform(x_test_stem_tfidf)

In [186]:
stem_stopkept_svd_original_tfidf=TruncatedSVD(n_components=50)
x_train_stem_stopkept_originald_tfidf_svd=stem_stopkept_svd_original_tfidf.fit_transform(X_train_stem_stopkept_tfidf)
x_test_stem_stopkept_originald_tfidf_svd=stem_stopkept_svd_original_tfidf.transform(x_test_stem_stopkept_tfidf)

In [187]:
lemma_svd_original_tfidf=TruncatedSVD(n_components=50)
x_train_lemma_originald_tfidf_svd=lemma_svd_original_tfidf.fit_transform(X_train_lemma_tfidf)
x_test_lemma_originald_tfidf_svd=lemma_svd_original_tfidf.transform(x_test_lemma_tfidf)

In [188]:
lemma_stopkept_svd_original_tfidf=TruncatedSVD(n_components=50)
x_train_lemma_stopkept_originald_tfidf_svd=lemma_stopkept_svd_original_tfidf.fit_transform(X_train_lemma_stopkept_tfidf)
x_test_lemma_stopkept_originald_tfidf_svd=lemma_stopkept_svd_original_tfidf.transform(x_test_lemma_stopkept_tfidf)

#### apply HMM

In [189]:
lengths_train = [1] * x_train_stem_originald_tfidf_svd.shape[0]
lengths_test = [1] * x_test_stem_originald_tfidf_svd.shape[0]

##### stem

In [190]:
hmm_stem_original_tfidf.fit(x_train_stem_originald_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [191]:
y_pred_stem_tfidf_original=hmm_stem_original_tfidf.predict(x_train_stem_originald_tfidf_svd,lengths=lengths_train)
accuracy_hmm_stem_tfidf_original = accuracy_score(y_train, y_pred_stem_tfidf_original)   
print(f"training HMM BoW Accuracy with length : {accuracy_hmm_stem_tfidf_original}")

training HMM BoW Accuracy with length : 0.08564814814814815


In [192]:
y_pred_stem_tfidf_original=hmm_stem_original_tfidf.predict(x_test_stem_originald_tfidf_svd,lengths=lengths_test)
accuracy_hmm_stem_tfidf_original = accuracy_score(y_test, y_pred_stem_tfidf_original)   
print(f"testing HMM BoW Accuracy with length : {accuracy_hmm_stem_tfidf_original}")

testing HMM BoW Accuracy with length : 0.08564610428349839


In [193]:
y_pred_stem_tfidf_original=hmm_stem_original_tfidf.predict(x_test_stem_originald_tfidf_svd)
accuracy_hmm_stem_tfidf_original = accuracy_score(y_test, y_pred_stem_tfidf_original)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_tfidf_original}")

testing HMM BoW Accuracy : 0.04593723899296027


In [195]:
results_Bow_orginal['(tf-idf) HMM stem+StopWords removed']=0.08564610428349839

##### stem +stopwords kept

In [196]:
hmm_stem_stopkept_original_tfidf.fit(x_train_stem_stopkept_originald_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [199]:
y_pred_stem_stopkept_tfidf_original=hmm_stem_stopkept_original_tfidf.predict(x_train_stem_stopkept_originald_tfidf_svd,lengths=lengths_train)
accuracy_hmm_stem_stopkept_tfidf_original = accuracy_score(y_train, y_pred_stem_stopkept_tfidf_original)   
print(f"training HMM BoW Accuracy with length : {accuracy_hmm_stem_stopkept_tfidf_original}")

training HMM BoW Accuracy with length : 0.16991814623902252


In [197]:
y_pred_stem_stopkept_tfidf_original=hmm_stem_stopkept_original_tfidf.predict(x_test_stem_stopkept_originald_tfidf_svd,lengths=lengths_test)
accuracy_hmm_stem_stopkept_tfidf_original = accuracy_score(y_test, y_pred_stem_stopkept_tfidf_original)   
print(f"testing HMM BoW Accuracy with length : {accuracy_hmm_stem_stopkept_tfidf_original}")

testing HMM BoW Accuracy with length : 0.1699081255220141


In [198]:
y_pred_stem_stopkept_tfidf_original=hmm_stem_stopkept_original_tfidf.predict(x_test_stem_stopkept_originald_tfidf_svd)
accuracy_hmm_stem_stopkept_tfidf_original = accuracy_score(y_test, y_pred_stem_stopkept_tfidf_original)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_original}")

testing HMM BoW Accuracy : 0.048944040090681304


In [200]:
results_Bow_orginal['(tf-idf) HMM stem+StopWords kept']=0.1699081255220141

##### lemma

In [209]:
hmm_lemma_original_tfidf.fit(x_train_lemma_originald_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [213]:
y_pred_lemma_tfidf_original=hmm_lemma_original_tfidf.predict(x_train_lemma_originald_tfidf_svd)
accuracy_hmm_lemma_tfidf_original = accuracy_score(y_train, y_pred_lemma_tfidf_original)   
print(f"training HMM BoW Accuracy with length : {accuracy_hmm_lemma_tfidf_original}")

training HMM BoW Accuracy with length : 0.06450458190148912


In [211]:
y_pred_lemma_tfidf_original=hmm_lemma_original_tfidf.predict(x_test_lemma_originald_tfidf_svd,lengths=lengths_test)
accuracy_hmm_lemma_tfidf_original = accuracy_score(y_test, y_pred_lemma_tfidf_original)   
print(f"testing HMM BoW Accuracy with length : {accuracy_hmm_lemma_tfidf_original}")

testing HMM BoW Accuracy with length : 0.0285884739291254


In [212]:
y_pred_lemma_tfidf_original=hmm_lemma_original_tfidf.predict(x_test_lemma_originald_tfidf_svd)
accuracy_hmm_lemma_tfidf_original = accuracy_score(y_test, y_pred_lemma_tfidf_original)   
print(f"testing HMM BoW Accuracy with length : {accuracy_hmm_lemma_tfidf_original}")

testing HMM BoW Accuracy with length : 0.06299964204748837


In [214]:
results_Bow_orginal['(tf-idf) HMM lemma+StopWords removed']=0.06299964204748837

##### lemma + stopwords kept

In [201]:
hmm_lemma_stopkept_original_tfidf.fit(x_train_lemma_stopkept_originald_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [205]:
y_pred_lemma_stopkept_tfidf_original=hmm_lemma_stopkept_original_tfidf.predict(x_train_lemma_stopkept_originald_tfidf_svd)
accuracy_hmm_lemma_stopkept_tfidf_original = accuracy_score(y_train, y_pred_lemma_stopkept_tfidf_original)   
print(f"training HMM BoW Accuracy with length : {accuracy_hmm_lemma_stopkept_tfidf_original}")

training HMM BoW Accuracy with length : 0.05982722413134784


In [203]:
y_pred_lemma_stopkept_tfidf_original=hmm_lemma_stopkept_original_tfidf.predict(x_test_lemma_stopkept_originald_tfidf_svd,lengths=lengths_test)
accuracy_hmm_lemma_stopkept_tfidf_original = accuracy_score(y_test, y_pred_lemma_stopkept_tfidf_original)   
print(f"testing HMM BoW Accuracy with length : {accuracy_hmm_lemma_stopkept_tfidf_original}")

testing HMM BoW Accuracy with length : 0.0


In [204]:
y_pred_lemma_stopkept_tfidf_original=hmm_lemma_stopkept_original_tfidf.predict(x_test_lemma_stopkept_originald_tfidf_svd)
accuracy_hmm_lemma_stopkept_tfidf_original = accuracy_score(y_test, y_pred_lemma_stopkept_tfidf_original)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_lemma_stopkept_tfidf_original}")

testing HMM BoW Accuracy : 0.05956329793580718


In [218]:
results_Bow_orginal['(tf-idf) HMM lemma+StopWords kept']=0.05956329793580718

### MLP With dimension reduction

In [222]:
orginal_stem_MLP_tfidf=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
orginal_stem_stopkept_MLP_tfidf=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
orginal_lemma_MLP_tfidf=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
orginal_lemma_stopkept_MLP_tfidf=MLPClassifier(hidden_layer_sizes=(80,40), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)

#### stem

In [223]:
orginal_stem_MLP_tfidf.fit(x_train_stem_originald_tfidf_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [224]:
y_pred_train=orginal_stem_MLP_tfidf.predict(x_train_stem_originald_tfidf_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.596446639938908
              precision    recall  f1-score   support

           1       0.70      0.65      0.67     28481
           2       0.61      0.23      0.34     14356
           3       0.53      0.33      0.41     13889
           4       0.68      0.28      0.40      7920
           5       0.76      0.51      0.61      7851
           6       0.77      0.17      0.28      5077
           7       0.60      0.32      0.42      5072
           8       0.81      0.06      0.11      4793
           9       0.69      0.02      0.03      4061
          10       0.56      0.85      0.68     76116

    accuracy                           0.60    167616
   macro avg       0.67      0.34      0.39    167616
weighted avg       0.62      0.60      0.56    167616



In [225]:
y_pred_test=orginal_stem_MLP_tfidf.predict(x_test_stem_originald_tfidf_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.5862307600524997
              precision    recall  f1-score   support

           1       0.69      0.64      0.66      7120
           2       0.59      0.23      0.33      3589
           3       0.50      0.32      0.39      3473
           4       0.67      0.27      0.39      1980
           5       0.73      0.49      0.59      1963
           6       0.77      0.17      0.28      1269
           7       0.60      0.29      0.39      1268
           8       0.64      0.05      0.09      1198
           9       0.64      0.02      0.03      1016
          10       0.56      0.84      0.67     19029

    accuracy                           0.59     41905
   macro avg       0.64      0.33      0.38     41905
weighted avg       0.60      0.59      0.55     41905



In [226]:
results_Bow_orginal['(tf-idf) MLP stem+stopwords removed']=0.5862307600524997

#### stem +stopwords kept

In [227]:
orginal_stem_stopkept_MLP_tfidf.fit(x_train_stem_stopkept_originald_tfidf_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [228]:
y_pred_train=orginal_stem_stopkept_MLP_tfidf.predict(x_train_stem_stopkept_originald_tfidf_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.5967389747995419
              precision    recall  f1-score   support

           1       0.70      0.66      0.68     28481
           2       0.60      0.25      0.35     14356
           3       0.53      0.31      0.39     13889
           4       0.67      0.29      0.41      7920
           5       0.70      0.54      0.61      7851
           6       0.74      0.17      0.28      5077
           7       0.59      0.34      0.43      5072
           8       0.71      0.06      0.12      4793
           9       0.62      0.02      0.03      4061
          10       0.57      0.84      0.68     76116

    accuracy                           0.60    167616
   macro avg       0.64      0.35      0.40    167616
weighted avg       0.61      0.60      0.56    167616



In [229]:
y_pred_test=orginal_stem_stopkept_MLP_tfidf.predict(x_test_stem_stopkept_originald_tfidf_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.5841307719842501
              precision    recall  f1-score   support

           1       0.68      0.64      0.66      7120
           2       0.56      0.23      0.33      3589
           3       0.49      0.30      0.38      3473
           4       0.67      0.28      0.40      1980
           5       0.68      0.52      0.59      1963
           6       0.72      0.17      0.27      1269
           7       0.59      0.31      0.40      1268
           8       0.59      0.06      0.10      1198
           9       0.64      0.02      0.03      1016
          10       0.56      0.83      0.67     19029

    accuracy                           0.58     41905
   macro avg       0.62      0.34      0.38     41905
weighted avg       0.59      0.58      0.55     41905



In [230]:
results_Bow_orginal['(tf-idf) MLP stem+stopwords kept']=0.5841307719842501

#### lemma

In [231]:
orginal_lemma_MLP_tfidf.fit(x_train_lemma_originald_tfidf_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [232]:
y_pred_train=orginal_lemma_MLP_tfidf.predict(x_train_lemma_originald_tfidf_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.5962437953417334
              precision    recall  f1-score   support

           1       0.69      0.65      0.67     28481
           2       0.57      0.27      0.36     14356
           3       0.56      0.29      0.39     13889
           4       0.62      0.28      0.39      7920
           5       0.71      0.55      0.62      7851
           6       0.74      0.19      0.30      5077
           7       0.63      0.30      0.41      5072
           8       0.81      0.06      0.11      4793
           9       0.51      0.02      0.04      4061
          10       0.57      0.84      0.68     76116

    accuracy                           0.60    167616
   macro avg       0.64      0.35      0.40    167616
weighted avg       0.61      0.60      0.56    167616



In [233]:
y_pred_test=orginal_lemma_MLP_tfidf.predict(x_test_lemma_originald_tfidf_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.5851807660183749
              precision    recall  f1-score   support

           1       0.68      0.64      0.66      7120
           2       0.54      0.25      0.34      3589
           3       0.55      0.29      0.38      3473
           4       0.61      0.28      0.38      1980
           5       0.69      0.52      0.59      1963
           6       0.74      0.18      0.29      1269
           7       0.62      0.27      0.38      1268
           8       0.73      0.05      0.09      1198
           9       0.40      0.02      0.03      1016
          10       0.56      0.83      0.67     19029

    accuracy                           0.59     41905
   macro avg       0.61      0.33      0.38     41905
weighted avg       0.59      0.59      0.55     41905



In [234]:
results_Bow_orginal['(tf-idf) MLP lemma+stopwords removed']=0.5851807660183749

#### lemma +stopwords kept

In [235]:
orginal_lemma_stopkept_MLP_tfidf.fit(x_train_lemma_stopkept_originald_tfidf_svd,y_train)

MLPClassifier(alpha=0.0005, batch_size=256, early_stopping=True,
              hidden_layer_sizes=(80, 40), max_iter=1000, random_state=42)

In [236]:
y_pred_train=orginal_lemma_stopkept_MLP_tfidf.predict(x_train_lemma_stopkept_originald_tfidf_svd)
accuracy=accuracy_score(y_train, y_pred_train)
print(f"MLP accuracy for training:{accuracy}")
print(classification_report(y_train, y_pred_train))

MLP accuracy for training:0.5906715349369989
              precision    recall  f1-score   support

           1       0.68      0.66      0.67     28481
           2       0.59      0.23      0.33     14356
           3       0.56      0.28      0.37     13889
           4       0.62      0.26      0.37      7920
           5       0.71      0.51      0.60      7851
           6       0.72      0.20      0.32      5077
           7       0.66      0.28      0.39      5072
           8       0.78      0.06      0.11      4793
           9       0.51      0.01      0.02      4061
          10       0.56      0.84      0.67     76116

    accuracy                           0.59    167616
   macro avg       0.64      0.33      0.39    167616
weighted avg       0.61      0.59      0.55    167616



In [237]:
y_pred_test=orginal_lemma_stopkept_MLP_tfidf.predict(x_test_lemma_stopkept_originald_tfidf_svd)
accuracy=accuracy_score(y_test, y_pred_test)
print(f"MLP accuracy for testing:{accuracy}")
print(classification_report(y_test, y_pred_test))

MLP accuracy for testing:0.5827228254384919
              precision    recall  f1-score   support

           1       0.68      0.64      0.66      7120
           2       0.56      0.22      0.32      3589
           3       0.53      0.27      0.36      3473
           4       0.62      0.26      0.36      1980
           5       0.70      0.50      0.59      1963
           6       0.68      0.20      0.31      1269
           7       0.67      0.26      0.37      1268
           8       0.65      0.05      0.09      1198
           9       0.35      0.01      0.01      1016
          10       0.56      0.84      0.67     19029

    accuracy                           0.58     41905
   macro avg       0.60      0.32      0.37     41905
weighted avg       0.59      0.58      0.54     41905



In [238]:
results_Bow_orginal['(tf-idf) MLP lemma+stopwords kept']=0.5827228254384919

## Save results

In [239]:
results_Bow_orginal

{'SVC_BoW stem+StopWords kept': 0.6616155590025057,
 'MultinomialNB_BoW stem+StopWords kept': 0.6672234816847632,
 'SVC_BoW Lemma+StopWords kept': 0.6605894284691565,
 'MultinomialNB_BoW Lemma+StopWords kept': 0.6684643837250924,
 'SVC_BoW stem+StopWords removed': 0.6614007874955256,
 'MultinomialNB_BoW stem+StopWords removed': 0.6731177663763274,
 'SVC_BoW lemma+StopWords removed': 0.6615439685001789,
 'MultinomialNB_BoW lemma+StopWords removed': 0.6737859443980432,
 'HMM stem+StopWords removed': 0.04331225390764825,
 'HMM stem+StopWords kept': 0.1699081255220141,
 'HMM lemma+StopWords kept': 0.06261782603507934,
 'HMM lemma+StopWords removed': 0.06369168356997972,
 'MLP stem+stopwords removed': 0.5851569025175993,
 'MLP stem+stopwords kept': 0.546498031261186,
 'MLP lemma+stopwords removed': 0.5852523565207016,
 'MLP lemma+stopwords kept': 0.5464503042596349,
 '(tf-idf) MultinomialNB_BoW stem+StopWords removed': 0.6731177663763274,
 '(tf-idf) SVC_BoW stem+StopWords removed': 0.661400

In [240]:
joblib.dump(results_Bow_orginal,'results_Bow_original')

['results_Bow_original']

In [183]:
results_Bow_orginal=joblib.load('results_Bow_original')
results_Bow_orginal

{'SVC_BoW stem+StopWords kept': 0.6616155590025057,
 'MultinomialNB_BoW stem+StopWords kept': 0.6672234816847632,
 'SVC_BoW Lemma+StopWords kept': 0.6605894284691565,
 'MultinomialNB_BoW Lemma+StopWords kept': 0.6684643837250924,
 'SVC_BoW stem+StopWords removed': 0.6614007874955256,
 'MultinomialNB_BoW stem+StopWords removed': 0.6731177663763274,
 'SVC_BoW lemma+StopWords removed': 0.6615439685001789,
 'MultinomialNB_BoW lemma+StopWords removed': 0.6737859443980432,
 'HMM stem+StopWords removed': 0.04331225390764825,
 'HMM stem+StopWords kept': 0.1699081255220141,
 'HMM lemma+StopWords kept': 0.06261782603507934,
 'HMM lemma+StopWords removed': 0.06369168356997972,
 'MLP stem+stopwords removed': 0.5851569025175993,
 'MLP stem+stopwords kept': 0.546498031261186,
 'MLP lemma+stopwords removed': 0.5852523565207016,
 'MLP lemma+stopwords kept': 0.5464503042596349,
 '(tf-idf) MultinomialNB_BoW stem+StopWords removed': 0.6731177663763274,
 '(tf-idf) SVC_BoW stem+StopWords removed': 0.661400

In [241]:
with open("results_Bow_orginal.txt", "w", encoding="utf-8") as f:
    for key, value in results_Bow_orginal.items():
        f.write(f"{key}: {value}\n")

# ______________________________________________________________________________________

# resampled Data (undersample)

In [12]:
# results_Bow_undersample={}
results_Bow_undersample=joblib.load('results_Bow_undersample')
results_Bow_undersample

{'undersample SVM stem+StopWords removed': 0.6263328141847371,
 'undersample MultinomialNB stem+StopWords removed': 0.6012938780400143,
 'undersample MLP stem+StopWords removed': 0.6314843656403498,
 'undersample SVM stem+StopWords kept': 0.6225590032346952,
 'undersample MultinomialNB stem+StopWords kept': 0.5940457649454894,
 'undersample MLP stem+StopWords kept': 0.6299868216125554,
 'undersample SVM lemma+StopWords removed': 0.6224391997124715,
 'undersample MultinomialNB lemma+StopWords removed': 0.5948843896010543,
 'undersample MLP lemma+StopWords removed': 0.6245357613513838,
 'undersample SVM lemma+StopWords kept': 0.6184856834790943,
 'undersample MultinomialNB lemma+StopWords kept': 0.5880555888343117,
 'undersample MLP lemma+StopWords kept': 0.6229783155624775,
 'undersample HMM stem+StopWords removed': 0.0770935665508566,
 'undersample HMM stem+StopWords kept': 0.11980352222355337,
 'undersample HMM lemma+StopWords removed': 0.08649814304540554,
 'undersample HMM lemma+Sto

## BoW feature Extraction

### with countvectorizer

In [30]:
vectorizer_stem_undersample=CountVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=True),max_features=1000)
x_train_stem_undersampled=vectorizer_stem_undersample.fit_transform(x_train_undersampled)
x_test_stem_undersampled=vectorizer_stem_undersample.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
vectorizer_stem_stopkept_undersample=CountVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=False),max_features=1000)
x_train_stem_stopkept_undersampled=vectorizer_stem_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_stem_stopkept_undersampled=vectorizer_stem_stopkept_undersample.transform(x_test_undersampled)

In [15]:
vectorizer_lemma_undersample=CountVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=True),max_features=1000)
x_train_lemma_undersampled=vectorizer_lemma_undersample.fit_transform(x_train_undersampled)
x_test_lemma_undersampled=vectorizer_lemma_undersample.transform(x_test_undersampled)

In [16]:
vectorizer_lemma_stopkept_undersample=CountVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=False),max_features=1000)
x_train_lemma_stopkept_undersampled=vectorizer_lemma_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_lemma_stopkept_undersampled=vectorizer_lemma_stopkept_undersample.transform(x_test_undersampled)

### with tf-idf vectorizer

In [13]:
tfidf_stem_undersample=TfidfVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=True),max_features=1000)
x_train_stem_undersampled_tfidf=tfidf_stem_undersample.fit_transform(x_train_undersampled)
x_test_stem_undersampled_tfidf=tfidf_stem_undersample.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
tfidf_stem_stopkept_undersample=TfidfVectorizer(tokenizer=lambda x : stem(x,remove_stopwords=False),max_features=1000)
x_train_stem_stopkept_undersampled_tfidf=tfidf_stem_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_stem_stopkept_undersampled_tfidf=tfidf_stem_stopkept_undersample.transform(x_test_undersampled)

In [15]:
tfidf_lemma_undersample=TfidfVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=True),max_features=1000)
x_train_lemma_undersampled_tfidf=tfidf_lemma_undersample.fit_transform(x_train_undersampled)
x_test_lemma_undersampled_tfidf=tfidf_lemma_undersample.transform(x_test_undersampled)

In [16]:
tfidf_lemma_stopkept_undersample=TfidfVectorizer(tokenizer=lambda x : lemma(x,remove_stopwords=False),max_features=1000)
x_train_lemma_stopkept_undersampled_tfidf=tfidf_lemma_stopkept_undersample.fit_transform(x_train_undersampled)
x_test_lemma_stopkept_undersampled_tfidf=tfidf_lemma_stopkept_undersample.transform(x_test_undersampled)

In [17]:
x_train_lemma_stopkept_undersampled_tfidf.shape

(66774, 1000)

## Models

In [18]:
models_Bow_undersample={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(512,256,128), max_iter=1000,activation='relu',solver='adam',early_stopping=True,alpha=0.0005,batch_size=256,random_state=42)
}

## with countvectorizer

#### stem 

In [24]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_stem_undersampled) #removed
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_undersampled) #removed
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" stem+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6263328141847371
              precision    recall  f1-score   support

           1       0.69      0.71      0.70      2000
           2       0.47      0.73      0.57      2000
           3       0.61      0.72      0.66      2000
           4       0.70      0.66      0.68      1980
           5       0.83      0.78      0.80      1963
           6       0.88      0.60      0.72      1269
           7       0.74      0.60      0.66      1268
           8       0.64      0.44      0.52      1198
           9       0.73      0.50      0.59      1016
          10       0.37      0.37      0.37      2000

    accuracy                           0.63     16694
   macro avg       0.66      0.61      0.63     16694
weighted avg       0.65      0.63      0.63     16694

Results for SVM Training: accuracy=0.765941833647827
--------------------------------------------------
Results for MultinomialNB: accuracy=0.6012938780400143
              precision    recall  f1

#### stem + stopwords kept

In [35]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_stopkept_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_stem_stopkept_undersampled) 
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_stopkept_undersampled) 
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" stem+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6225590032346952
              precision    recall  f1-score   support

           1       0.68      0.68      0.68      2000
           2       0.47      0.72      0.57      2000
           3       0.60      0.72      0.66      2000
           4       0.70      0.68      0.69      1980
           5       0.83      0.78      0.80      1963
           6       0.90      0.60      0.72      1269
           7       0.75      0.62      0.68      1268
           8       0.62      0.39      0.48      1198
           9       0.74      0.48      0.59      1016
          10       0.35      0.38      0.37      2000

    accuracy                           0.62     16694
   macro avg       0.66      0.61      0.62     16694
weighted avg       0.65      0.62      0.62     16694

Results for SVM Training: accuracy=0.7844071045616557
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5940457649454894
              precision    recall  f

#### lemma

In [38]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_undersampled) 
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_undersampled) 
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" lemma+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6224391997124715
              precision    recall  f1-score   support

           1       0.69      0.71      0.70      2000
           2       0.44      0.73      0.55      2000
           3       0.61      0.71      0.66      2000
           4       0.73      0.65      0.69      1980
           5       0.83      0.78      0.80      1963
           6       0.89      0.60      0.72      1269
           7       0.74      0.60      0.66      1268
           8       0.63      0.40      0.49      1198
           9       0.74      0.50      0.60      1016
          10       0.36      0.36      0.36      2000

    accuracy                           0.62     16694
   macro avg       0.67      0.61      0.62     16694
weighted avg       0.65      0.62      0.63     16694

Results for SVM Training: accuracy=0.7523587024889927
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5948843896010543
              precision    recall  f

#### lemma + stopwords kept

In [40]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_stopkept_undersampled, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_stopkept_undersampled) 
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_stopkept_undersampled) 
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['undersample '+model_name+" lemma+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6184856834790943
              precision    recall  f1-score   support

           1       0.69      0.68      0.68      2000
           2       0.45      0.73      0.56      2000
           3       0.60      0.73      0.66      2000
           4       0.70      0.68      0.69      1980
           5       0.84      0.77      0.80      1963
           6       0.90      0.59      0.71      1269
           7       0.76      0.60      0.67      1268
           8       0.63      0.38      0.47      1198
           9       0.76      0.48      0.59      1016
          10       0.35      0.38      0.36      2000

    accuracy                           0.62     16694
   macro avg       0.67      0.60      0.62     16694
weighted avg       0.65      0.62      0.62     16694

Results for SVM Training: accuracy=0.7733249468355947
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5880555888343117
              precision    recall  f

In [41]:
results_Bow_undersample

{'undersample SVM stem+StopWords removed': 0.6263328141847371,
 'undersample MultinomialNB stem+StopWords removed': 0.6012938780400143,
 'undersample MLP stem+StopWords removed': 0.6314843656403498,
 'undersample SVM stem+StopWords kept': 0.6225590032346952,
 'undersample MultinomialNB stem+StopWords kept': 0.5940457649454894,
 'undersample MLP stem+StopWords kept': 0.6299868216125554,
 'undersample SVM lemma+StopWords removed': 0.6224391997124715,
 'undersample MultinomialNB lemma+StopWords removed': 0.5948843896010543,
 'undersample MLP lemma+StopWords removed': 0.6245357613513838,
 'undersample SVM lemma+StopWords kept': 0.6184856834790943,
 'undersample MultinomialNB lemma+StopWords kept': 0.5880555888343117,
 'undersample MLP lemma+StopWords kept': 0.6229783155624775}

### HMM

In [43]:
hmm_stem_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_stem_stopkept_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_stopkept_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,50)

In [44]:
stem_svd_undersample=TruncatedSVD(n_components=50)
x_train_stem_undersampled_svd=stem_svd_undersample.fit_transform(x_train_stem_undersampled)
x_test_stem_undersampled_svd=stem_svd_undersample.transform(x_test_stem_undersampled)

In [45]:
stem_stopkept_svd_undersample=TruncatedSVD(n_components=50)
x_train_stem_stopkept_undersampled_svd=stem_stopkept_svd_undersample.fit_transform(x_train_stem_stopkept_undersampled)
x_test_stem_stopkept_undersampled_svd=stem_stopkept_svd_undersample.transform(x_test_stem_stopkept_undersampled)

In [46]:
lemma_svd_undersample=TruncatedSVD(n_components=50)
x_train_lemma_undersampled_svd=lemma_svd_undersample.fit_transform(x_train_lemma_undersampled)
x_test_lemma_undersampled_svd=lemma_svd_undersample.transform(x_test_lemma_undersampled)

In [47]:
lemma_stopkept_svd_undersample=TruncatedSVD(n_components=50)
x_train_lemma_stopkept_undersampled_svd=lemma_stopkept_svd_undersample.fit_transform(x_train_lemma_stopkept_undersampled)
x_test_lemma_stopkept_undersampled_svd=lemma_stopkept_svd_undersample.transform(x_test_lemma_stopkept_undersampled)

In [48]:
x_train_stem_undersampled_svd.shape

(66774, 50)

#### apply HMM

In [50]:
lengths_train = [1] * x_train_stem_undersampled_svd.shape[0]
lengths_test = [1] * x_test_stem_undersampled_svd.shape[0]

##### stem

In [51]:
hmm_stem_undersample.fit(x_train_stem_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [53]:
y_pred_stem_undersample=hmm_stem_undersample.predict(x_train_stem_undersampled_svd,lengths=lengths_train)
accuracy_hmm_stem_undersample = accuracy_score(y_train_undersampled, y_pred_stem_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

training HMM BoW Accuracy with lengths: 0.07177943510947375


In [54]:
y_pred_stem_undersample=hmm_stem_undersample.predict(x_test_stem_undersampled_svd,lengths=lengths_test)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy with lengths: 0.07176230981190847


In [55]:
y_pred_stem_undersample=hmm_stem_undersample.predict(x_test_stem_undersampled_svd)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy: 0.0770935665508566


In [57]:
results_Bow_undersample['undersample HMM stem+StopWords removed']=0.0770935665508566

##### stem + stopwords kept

In [59]:
hmm_stem_stopkept_undersample.fit(x_train_stem_stopkept_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [61]:
y_pred_stem_stopkept_undersample=hmm_stem_stopkept_undersample.predict(x_train_stem_stopkept_undersampled_svd,lengths=lengths_train)
accuracy_hmm_stem_stopkept_undersample = accuracy_score(y_train_undersampled, y_pred_stem_stopkept_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_stopkept_undersample}")

training HMM BoW Accuracy with lengths: 0.11980711055201126


In [63]:
y_pred_stem_stopkept_undersample=hmm_stem_stopkept_undersample.predict(x_test_stem_stopkept_undersampled_svd,lengths=lengths_test)
accuracy_hmm_stem_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_stopkept_undersample}")

testing HMM BoW Accuracy with lengths: 0.11980352222355337


In [64]:
y_pred_stem_stopkept_undersample=hmm_stem_stopkept_undersample.predict(x_test_stem_stopkept_undersampled_svd)
accuracy_hmm_stem_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_stopkept_undersample}")

testing HMM BoW Accuracy: 0.09230861387324787


In [65]:
results_Bow_undersample['undersample HMM stem+StopWords kept']=0.11980352222355337

##### lemma

In [68]:
hmm_lemma_undersample.fit(x_train_lemma_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [69]:
y_pred_lemma_undersample=hmm_lemma_undersample.predict(x_train_lemma_undersampled_svd,lengths=lengths_train)
accuracy_hmm_lemma_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_undersample}")

training HMM BoW Accuracy with lengths: 0.07177943510947375


In [70]:
y_pred_lemma_undersample=hmm_lemma_undersample.predict(x_test_lemma_undersampled_svd,lengths=lengths_test)
accuracy_hmm_lemma_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_undersample}")

testing HMM BoW Accuracy with lengths: 0.07176230981190847


In [71]:
y_pred_lemma_undersample=hmm_lemma_undersample.predict(x_test_lemma_undersampled_svd)
accuracy_hmm_lemma_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_lemma_undersample}")

testing HMM BoW Accuracy: 0.08649814304540554


In [83]:
results_Bow_undersample['undersample HMM lemma+StopWords removed']=0.08649814304540554

##### lemma + stopwords kept

In [73]:
hmm_lemma_stopkept_undersample.fit(x_train_lemma_stopkept_undersampled_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [74]:
y_pred_lemma_stopkept_undersample=hmm_lemma_stopkept_undersample.predict(x_train_lemma_stopkept_undersampled_svd,lengths=lengths_train)
accuracy_hmm_lemma_stopkept_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_stopkept_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_stopkept_undersample}")

training HMM BoW Accuracy with lengths: 0.0


In [75]:
y_pred_lemma_stopkept_undersample=hmm_lemma_stopkept_undersample.predict(x_test_lemma_stopkept_undersampled_svd,lengths=lengths_test)
accuracy_hmm_lemma_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_lemma_stopkept_undersample}")

testing HMM BoW Accuracy with lengths: 0.0


In [76]:
y_pred_lemma_stopkept_undersample=hmm_lemma_stopkept_undersample.predict(x_test_lemma_stopkept_undersampled_svd)
accuracy_hmm_lemma_stopkept_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_lemma_stopkept_undersample}")

testing HMM BoW Accuracy: 0.098178986462202


In [80]:
results_Bow_undersample['undersample HMM lemma+StopWords kept']=0.098178986462202

#### save the results of HMM

In [84]:
results_Bow_undersample

{'undersample SVM stem+StopWords removed': 0.6263328141847371,
 'undersample MultinomialNB stem+StopWords removed': 0.6012938780400143,
 'undersample MLP stem+StopWords removed': 0.6314843656403498,
 'undersample SVM stem+StopWords kept': 0.6225590032346952,
 'undersample MultinomialNB stem+StopWords kept': 0.5940457649454894,
 'undersample MLP stem+StopWords kept': 0.6299868216125554,
 'undersample SVM lemma+StopWords removed': 0.6224391997124715,
 'undersample MultinomialNB lemma+StopWords removed': 0.5948843896010543,
 'undersample MLP lemma+StopWords removed': 0.6245357613513838,
 'undersample SVM lemma+StopWords kept': 0.6184856834790943,
 'undersample MultinomialNB lemma+StopWords kept': 0.5880555888343117,
 'undersample MLP lemma+StopWords kept': 0.6229783155624775,
 'undersample HMM stem+StopWords removed': 0.0770935665508566,
 'undersample HMM stem+StopWords kept': 0.11980352222355337,
 'undersample HMM lemma+StopWords removed': 0.08649814304540554,
 'undersample HMM lemma+Sto

In [85]:
joblib.dump(results_Bow_undersample,'results_Bow_undersample')

['results_Bow_undersample']

## with tf-idf

### stem

In [19]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_stem_undersampled_tfidf) #removed
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_undersampled_tfidf) #removed
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" stem+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6383131664070923
              precision    recall  f1-score   support

           1       0.69      0.72      0.70      2000
           2       0.51      0.71      0.59      2000
           3       0.61      0.72      0.66      2000
           4       0.69      0.69      0.69      1980
           5       0.83      0.79      0.81      1963
           6       0.89      0.61      0.72      1269
           7       0.73      0.64      0.68      1268
           8       0.61      0.46      0.52      1198
           9       0.72      0.52      0.60      1016
          10       0.39      0.40      0.39      2000

    accuracy                           0.64     16694
   macro avg       0.67      0.62      0.64     16694
weighted avg       0.65      0.64      0.64     16694

Results for SVM Training: accuracy=0.7903525324227993
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5993171199233257
              precision    recall  f

### stem + stopwords kept

In [20]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_stem_stopkept_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_stem_stopkept_undersampled_tfidf) #kept
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_stem_stopkept_undersampled_tfidf) #kept
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" stem+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6371151311848569
              precision    recall  f1-score   support

           1       0.67      0.71      0.69      2000
           2       0.52      0.69      0.59      2000
           3       0.61      0.73      0.67      2000
           4       0.71      0.71      0.71      1980
           5       0.82      0.79      0.80      1963
           6       0.90      0.61      0.72      1269
           7       0.72      0.67      0.69      1268
           8       0.60      0.43      0.51      1198
           9       0.72      0.51      0.60      1016
          10       0.38      0.39      0.38      2000

    accuracy                           0.64     16694
   macro avg       0.66      0.62      0.64     16694
weighted avg       0.65      0.64      0.64     16694

Results for SVM Training: accuracy=0.8133554976487855
--------------------------------------------------
Results for MultinomialNB: accuracy=0.593626452617707
              precision    recall  f1

### lemma

In [24]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_undersampled_tfidf) #removed
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_undersampled_tfidf) #removed
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" lemma+StopWords removed"] = accuracy

Results for SVM: accuracy=0.6307655445070085
              precision    recall  f1-score   support

           1       0.68      0.72      0.70      2000
           2       0.48      0.70      0.57      2000
           3       0.62      0.71      0.66      2000
           4       0.70      0.69      0.69      1980
           5       0.83      0.78      0.81      1963
           6       0.89      0.61      0.72      1269
           7       0.73      0.63      0.67      1268
           8       0.60      0.42      0.50      1198
           9       0.71      0.53      0.61      1016
          10       0.38      0.38      0.38      2000

    accuracy                           0.63     16694
   macro avg       0.66      0.62      0.63     16694
weighted avg       0.65      0.63      0.63     16694

Results for SVM Training: accuracy=0.7760355827118339
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5944051755121601
              precision    recall  f

### lemma + stopwords kept

In [27]:
for model_name, model in models_Bow_undersample.items():
    model.fit(x_train_lemma_stopkept_undersampled_tfidf, y_train_undersampled)
    y_pred = model.predict(x_test_lemma_stopkept_undersampled_tfidf) #kept
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    
    y_pred_train = model.predict(x_train_lemma_stopkept_undersampled_tfidf) #kept
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    print('-'*50)
    
    results_Bow_undersample['(tf-idf)undersample '+model_name+" lemma+StopWords kept"] = accuracy

Results for SVM: accuracy=0.6337606325625973
              precision    recall  f1-score   support

           1       0.68      0.70      0.69      2000
           2       0.50      0.71      0.59      2000
           3       0.61      0.73      0.67      2000
           4       0.70      0.71      0.71      1980
           5       0.84      0.78      0.81      1963
           6       0.89      0.61      0.72      1269
           7       0.74      0.63      0.68      1268
           8       0.60      0.41      0.49      1198
           9       0.73      0.51      0.60      1016
          10       0.37      0.39      0.38      2000

    accuracy                           0.63     16694
   macro avg       0.67      0.62      0.63     16694
weighted avg       0.65      0.63      0.64     16694

Results for SVM Training: accuracy=0.8020037739239824
--------------------------------------------------
Results for MultinomialNB: accuracy=0.5894333293398826
              precision    recall  f

### HMM

In [30]:
hmm_stem_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_stem_stopkept_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_stopkept_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,50)

In [31]:
stem_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_stem_undersampled_tfidf_svd=stem_svd_undersample_tfidf.fit_transform(x_train_stem_undersampled_tfidf)
x_test_stem_undersampled_tfidf_svd=stem_svd_undersample_tfidf.transform(x_test_stem_undersampled_tfidf)

In [32]:
stem_stopkept_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_stem_stopkept_undersampled_tfidf_svd=stem_stopkept_svd_undersample_tfidf.fit_transform(x_train_stem_stopkept_undersampled_tfidf)
x_test_stem_stopkept_undersampled_tfidf_svd=stem_stopkept_svd_undersample_tfidf.transform(x_test_stem_stopkept_undersampled_tfidf)

In [33]:
lemma_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_lemma_undersampled_tfidf_svd=lemma_svd_undersample_tfidf.fit_transform(x_train_lemma_undersampled_tfidf)
x_test_lemma_undersampled_tfidf_svd=lemma_svd_undersample_tfidf.transform(x_test_lemma_undersampled_tfidf)

In [34]:
lemma_stopkept_svd_undersample_tfidf=TruncatedSVD(n_components=50)
x_train_lemma_stopkept_undersampled_tfidf_svd=lemma_stopkept_svd_undersample_tfidf.fit_transform(x_train_lemma_stopkept_undersampled_tfidf)
x_test_lemma_stopkept_undersampled_tfidf_svd=lemma_stopkept_svd_undersample_tfidf.transform(x_test_lemma_stopkept_undersampled_tfidf)

#### apply HMM

In [35]:
lengths_train = [1] * x_train_stem_stopkept_undersampled_tfidf_svd.shape[0]
lengths_test = [1] * x_test_stem_stopkept_undersampled_tfidf_svd.shape[0]

In [37]:
lengths_train

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


##### stem

In [36]:
hmm_stem_undersample_tfidf.fit(x_train_stem_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [41]:
y_pred_stem_tfidf_undersample=hmm_stem_undersample_tfidf.predict(x_train_stem_undersampled_tfidf_svd,)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_tfidf_undersample}")

training HMM BoW Accuracy : 0.08810315392218528


In [39]:
y_pred_stem_tfidf_undersample=hmm_stem_undersample_tfidf.predict(x_test_stem_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy with lengths: 0.06086018928956511


In [40]:
y_pred_stem_tfidf_undersample=hmm_stem_undersample_tfidf.predict(x_test_stem_undersampled_tfidf_svd)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy: 0.09129028393434767


In [48]:
results_Bow_undersample['(tf-idf)undersample HMM stem+StopWords removed']=0.09129028393434767

##### stem + stopwords kept

In [42]:
hmm_stem_stopkept_undersample_tfidf.fit(x_train_stem_stopkept_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [43]:
y_pred_stem_stopkept_tfidf_undersample=hmm_stem_stopkept_undersample_tfidf.predict(x_train_stem_stopkept_undersampled_tfidf_svd,lengths=lengths_train)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

training HMM BoW Accuracy : 0.11762063078443706


In [44]:
y_pred_stem_stopkept_tfidf_undersample=hmm_stem_stopkept_undersample_tfidf.predict(x_test_stem_stopkept_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.11758715706241764


In [45]:
y_pred_stem_stopkept_tfidf_undersample=hmm_stem_stopkept_undersample_tfidf.predict(x_test_stem_stopkept_undersampled_tfidf_svd,)
accuracy_hmm_stem_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_stem_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.0938061579010423


In [46]:
results_Bow_undersample['(tf-idf)undersample HMM stem+StopWords kept']=0.11758715706241764

##### lemma

In [50]:
hmm_lemma_undersample_tfidf.fit(x_train_lemma_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [51]:
y_pred_lemma_tfidf_undersample=hmm_lemma_undersample_tfidf.predict(x_train_lemma_undersampled_tfidf_svd,lengths=lengths_train)
accuracy_hmm_lemma_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_lemma_tfidf_undersample}")

training HMM BoW Accuracy : 0.11757570311798005


In [52]:
y_pred_lemma_tfidf_undersample=hmm_lemma_undersample_tfidf.predict(x_test_lemma_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_lemma_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with length: {accuracy_hmm_lemma_tfidf_undersample}")

testing HMM BoW Accuracy with length: 0.11758715706241764


In [53]:
y_pred_lemma_tfidf_undersample=hmm_lemma_undersample_tfidf.predict(x_test_lemma_undersampled_tfidf_svd,)
accuracy_hmm_lemma_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_lemma_tfidf_undersample}")

testing HMM BoW Accuracy : 0.07391877321193244


In [54]:
results_Bow_undersample['(tf-idf)undersample HMM lemma+StopWords removed']=0.11758715706241764

##### lemma + stopwords kept

In [56]:
hmm_lemma_stopkept_undersample_tfidf.fit(x_train_lemma_stopkept_undersampled_tfidf_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [57]:
y_pred_lemma_stopkept_tfidf_undersample=hmm_lemma_stopkept_undersample_tfidf.predict(x_train_lemma_stopkept_undersampled_tfidf_svd,lengths=lengths_train)
accuracy_hmm_lemma_stopkept_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_lemma_stopkept_tfidf_undersample)   
print(f"training HMM BoW Accuracy with length : {accuracy_hmm_lemma_stopkept_tfidf_undersample}")

training HMM BoW Accuracy with length : 0.11980711055201126


In [58]:
y_pred_lemma_stopkept_tfidf_undersample=hmm_lemma_stopkept_undersample_tfidf.predict(x_test_lemma_stopkept_undersampled_tfidf_svd,lengths=lengths_test)
accuracy_hmm_lemma_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with length : {accuracy_hmm_lemma_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy with length : 0.11980352222355337


In [59]:
y_pred_lemma_stopkept_tfidf_undersample=hmm_lemma_stopkept_undersample_tfidf.predict(x_test_lemma_stopkept_undersampled_tfidf_svd)
accuracy_hmm_lemma_stopkept_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_lemma_stopkept_tfidf_undersample)   
print(f"testing HMM BoW Accuracy : {accuracy_hmm_lemma_stopkept_tfidf_undersample}")

testing HMM BoW Accuracy : 0.08506050077872289


In [67]:
results_Bow_undersample['(tf-idf)undersample HMM lemma+StopWords keptd']=0.11980352222355337

### save the results

In [61]:
joblib.dump(results_Bow_undersample,'results_Bow_undersample')

['results_Bow_undersample']

In [63]:
len(results_Bow_undersample)

32

In [66]:
with open("results_Bow_undersample.txt", "w", encoding="utf-8") as f:
    for key, value in results_Bow_undersample.items():
        f.write(f"{key}: {value}\n")

Although SVM achieved the highest training accuracy, it suffered from significant overfitting. In contrast, Multinomial Naive Bayes demonstrated strong generalization, while the MLP offered a balanced trade-off between model capacity and generalization.

# ______________________________________________________________________________________

# resampled Data (rosrus)

## BoW feature Extraction

In [10]:
vectorizer_stem_rosrus = CountVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=True))
x_train_stem_rosrus = vectorizer_stem_rosrus.fit_transform(x_train_bal)
x_test_stem_rosrus = vectorizer_stem_rosrus.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
vectorizer_stem_stopKept_rosrus = CountVectorizer(tokenizer=lambda x: stem(x, remove_stopwords=False))
x_train_stem_stopkept_rosrus = vectorizer_stem_stopKept_rosrus.fit_transform(x_train_bal)
x_test_stem_stopkept_rosrus = vectorizer_stem_stopKept_rosrus.transform(x_test_bal)

In [12]:
vectorizer_lemma_rosrus = CountVectorizer(tokenizer=lambda x:lemma(x, remove_stopwords=True))
x_train_lemma_rosrus = vectorizer_lemma_rosrus.fit_transform(x_train_bal)
x_test_lemma_rosrus = vectorizer_lemma_rosrus.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [13]:
vectorizer_lemma_stopkept_rosrus = CountVectorizer(tokenizer=lambda x: lemma(x, remove_stopwords=False))
x_train_lemma_stopkept_rosrus = vectorizer_lemma_stopkept_rosrus.fit_transform(x_train_bal)
x_test_lemma_stopkept_rosrus = vectorizer_lemma_stopkept_rosrus.transform(x_test_bal)

## Models (SVM & NB)

In [ ]:
# results_Bow_rosrus={}
results_Bow_rosrus

### Stem + stop words kept

In [37]:
for model_name, model in models_Bow.items():
    model.fit(x_train_stem_stopkept_rosrus, y_train_bal)
    y_pred = model.predict(x_test_stem_stopkept_rosrus) #kept
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_Bow_rosrus['rosrus '+model_name+" stem+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.45715308435747526
              precision    recall  f1-score   support

           1       0.64      0.77      0.70      7120
           2       0.32      0.84      0.46      3589
           3       0.36      0.85      0.50      3473
           4       0.46      0.79      0.58      1980
           5       0.53      0.82      0.64      1963
           6       0.47      0.66      0.55      1269
           7       0.52      0.76      0.62      1268
           8       0.26      0.63      0.37      1198
           9       0.53      0.68      0.59      1016
          10       0.97      0.07      0.13     19029

    accuracy                           0.46     41905
   macro avg       0.50      0.69      0.51     41905
weighted avg       0.70      0.46      0.37     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.559026369168357
              precision    recall  f1-score   support

           1       0.67    

### lemma + stopwords kept

In [39]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_lemma_stopkept_rosrus, y_train_bal)
    y_pred = model.predict(x_test_lemma_stopkept_rosrus) #kept
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)   
    results_Bow_rosrus['rosrus '+model_name+" Lemma+StopWords kept"] = accuracy

Results for SVC_BoW: accuracy=0.4530246987233027
              precision    recall  f1-score   support

           1       0.64      0.77      0.70      7120
           2       0.31      0.85      0.46      3589
           3       0.35      0.85      0.50      3473
           4       0.46      0.80      0.58      1980
           5       0.54      0.81      0.65      1963
           6       0.47      0.66      0.55      1269
           7       0.52      0.76      0.62      1268
           8       0.27      0.61      0.37      1198
           9       0.55      0.67      0.60      1016
          10       0.97      0.06      0.11     19029

    accuracy                           0.45     41905
   macro avg       0.51      0.68      0.51     41905
weighted avg       0.70      0.45      0.37     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.5595275026846438
              precision    recall  f1-score   support

           1       0.67    

### Stem + remove stop words

In [ ]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_stem_rosrus, y_train_bal)
    y_pred = model.predict(x_test_stem_rosrus) #removed
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)   
    results_Bow['rosrus '+model_name+" stem+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.46693711967545637
              precision    recall  f1-score   support

           1       0.64      0.78      0.70      7120
           2       0.33      0.83      0.47      3589
           3       0.37      0.85      0.51      3473
           4       0.48      0.80      0.60      1980
           5       0.52      0.82      0.64      1963
           6       0.46      0.67      0.55      1269
           7       0.51      0.77      0.62      1268
           8       0.26      0.64      0.37      1198
           9       0.53      0.70      0.60      1016
          10       0.96      0.08      0.15     19029

    accuracy                           0.47     41905
   macro avg       0.51      0.70      0.52     41905
weighted avg       0.70      0.47      0.39     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.5601479537048085
              precision    recall  f1-score   support

           1       0.67   

### Lemma + remove stop words

In [41]:
for model_name, model in models_Bow.items():    
    model.fit(x_train_lemma_rosrus, y_train_bal)
    y_pred = model.predict(x_test_lemma_rosrus) #removed
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)   
    results_Bow['rosrus '+model_name+" lemma+StopWords removed"] = accuracy

Results for SVC_BoW: accuracy=0.46404963608161315
              precision    recall  f1-score   support

           1       0.64      0.78      0.70      7120
           2       0.32      0.85      0.46      3589
           3       0.36      0.85      0.51      3473
           4       0.48      0.80      0.60      1980
           5       0.53      0.82      0.65      1963
           6       0.46      0.67      0.55      1269
           7       0.52      0.77      0.62      1268
           8       0.27      0.63      0.38      1198
           9       0.55      0.69      0.61      1016
          10       0.97      0.07      0.14     19029

    accuracy                           0.46     41905
   macro avg       0.51      0.69      0.52     41905
weighted avg       0.70      0.46      0.38     41905

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.5621763512707314
              precision    recall  f1-score   support

           1       0.67   

## HMM  (generative unsupervised)

hmm require dense data so use .toarray() is a solution but the shape (130000, 44211) is too large to allocate in memory

using dimension reduction : PCA also requires dense data(.toarray)--> same problem

In [29]:
hmm_BOW = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)

In [31]:
hmm_BOW.fit(x_train_lemma_rosrus.toarray())

MemoryError: Unable to allocate 42.8 GiB for an array with shape (130000, 44211) and data type float64

### dimension reduction

TruncatedSVD :faster and accept sparse and dense data

In [15]:
svd_stem_stopkept_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_stem_stopkept_rosrus=svd_stem_stopkept_rosrus.fit_transform(x_train_stem_stopkept_rosrus)
x_test_bow_svd_stem_stopkept_rosrus=svd_stem_stopkept_rosrus.transform(x_test_stem_stopkept_rosrus)

In [16]:
svd_stem_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_stem_rosrus=svd_stem_rosrus.fit_transform(x_train_stem_rosrus)
x_test_bow_svd_stem_rosrus=svd_stem_rosrus.transform(x_test_stem_rosrus)

In [17]:
svd_lemma_stopkept_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_lemma_stopkept_rosrus=svd_lemma_stopkept_rosrus.fit_transform(x_train_lemma_stopkept_rosrus)
x_test_bow_svd_lemma_stopkept_rosrus=svd_lemma_stopkept_rosrus.transform(x_test_lemma_stopkept_rosrus)

In [18]:
svd_lemma_rosrus=TruncatedSVD(n_components=50)
x_train_bow_svd_lemma_rosrus=svd_lemma_rosrus.fit_transform(x_train_lemma_rosrus)
x_test_bow_svd_lemma_rosrus=svd_lemma_rosrus.transform(x_test_lemma_rosrus)

In [48]:
hmm_stem_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_stem_stopkept_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_lemma_stopkept_bow_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi',  n_iter=100, random_state=42)

In [45]:
lengths_train = [1] * x_train_bow_svd_stem_rosrus.shape[0]
lengths_test = [1] * x_test_bow_svd_stem_rosrus.shape[0]

### stem + remove stopwords

In [49]:
hmm_stem_bow_rosrus.fit(x_train_bow_svd_stem_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [66]:
y_pred_stem_bow_rosrus=hmm_stem_bow_rosrus.predict(x_train_bow_svd_stem_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_stem_rosrus = accuracy_score(y_train_bal, y_pred_stem_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [101]:
y_pred_stem_bow_rosrus=hmm_stem_bow_rosrus.predict(x_test_bow_svd_stem_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_stem_rosrus = accuracy_score(y_test_bal, y_pred_stem_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_rosrus}")

testing HMM BoW Accuracy with lengths: 0.1699081255220141


In [52]:
y_pred_stem_bow_rosrus=hmm_stem_bow_rosrus.predict(x_test_bow_svd_stem_rosrus)
accuracy_hmm_BOW_stem_rosrus = accuracy_score(y_test_bal, y_pred_stem_bow_rosrus)   
print(f"HMM BoW Accuracy: {accuracy_hmm_BOW_stem_rosrus}")

HMM BoW Accuracy: 0.054026965755876385


In [92]:
results_Bow_rosrus['rosrus HMM_BoW stem+StopWords removed']=0.1699081255220141

### Stem + stopwords kept

In [58]:
hmm_stem_stopkept_bow_rosrus.fit(x_train_bow_svd_stem_stopkept_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [74]:
y_pred_stem_stopkept_bow_rosrus=hmm_stem_stopkept_bow_rosrus.predict(x_train_bow_svd_stem_stopkept_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_stem_stopkept_rosrus = accuracy_score(y_train_bal, y_pred_stem_stopkept_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_stopkept_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [75]:
y_pred_stem_stopkept_bow_rosrus=hmm_stem_stopkept_bow_rosrus.predict(x_test_bow_svd_stem_stopkept_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_stem_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_stem_stopkept_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_stopkept_rosrus}")

testing HMM BoW Accuracy with lengths: 0.082877938193533


In [76]:
y_pred_stem_stopkept_bow_rosrus=hmm_stem_stopkept_bow_rosrus.predict(x_test_bow_svd_stem_stopkept_rosrus)
accuracy_hmm_BOW_stem_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_stem_stopkept_bow_rosrus)   
print(f"HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_stem_stopkept_rosrus}")

HMM BoW Accuracy with lengths: 0.05901443741796922


In [93]:
results_Bow_rosrus['rosrus HMM_BoW stem+StopWords kept']=0.082877938193533

In [78]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533}

### lemma + remove stopwords

In [71]:
hmm_lemma_bow_rosrus.fit(x_train_bow_svd_lemma_rosrus)

Model is not converging.  Current: 9298625.125124881 is not greater than 9298625.801044906. Delta is -0.6759200245141983


GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [72]:
y_pred_lemma_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_train_bow_svd_lemma_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_lemma_rosrus = accuracy_score(y_train_bal, y_pred_lemma_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [73]:
y_pred_lemma_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_lemma_rosrus = accuracy_score(y_test_bal, y_pred_lemma_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_rosrus}")

testing HMM BoW Accuracy with lengths: 0.024245316787972794


In [81]:
y_pred_lemma_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_rosrus)
accuracy_hmm_BOW_lemma_rosrus = accuracy_score(y_test_bal, y_pred_lemma_bow_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_BOW_lemma_rosrus}")

testing HMM BoW Accuracy: 0.060350793461400785


In [ ]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533}

In [97]:
results_Bow_rosrus['rosrus HMM_BoW lemma+StopWords removed']=0.060350793461400785

### lemma + stop words kept

In [83]:
hmm_lemma_stopkept_bow_rosrus.fit(x_train_bow_svd_lemma_stopkept_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [84]:
y_pred_lemma_stopkept_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_train_bow_svd_lemma_stopkept_rosrus,lengths=lengths_train)
accuracy_hmm_BOW_lemma_stopkept_rosrus = accuracy_score(y_train_bal, y_pred_lemma_stopkept_bow_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_stopkept_rosrus}")

training HMM BoW Accuracy with lengths: 0.1


In [85]:
y_pred_lemma_stopkept_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_stopkept_rosrus,lengths=lengths_test)
accuracy_hmm_BOW_lemma_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_lemma_stopkept_bow_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_BOW_lemma_stopkept_rosrus}")

testing HMM BoW Accuracy with lengths: 0.024245316787972794


In [86]:
y_pred_lemma_stopkept_bow_rosrus=hmm_lemma_bow_rosrus.predict(x_test_bow_svd_lemma_stopkept_rosrus)
accuracy_hmm_BOW_lemma_stopkept_rosrus = accuracy_score(y_test_bal, y_pred_lemma_stopkept_bow_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_BOW_lemma_stopkept_rosrus}")

testing HMM BoW Accuracy: 0.022097601718172055


In [95]:
results_Bow_rosrus['rosrus HMM_BoW lemma+StopWords kept']=0.024245316787972794

## MLP with dimension reduction (130000, 50)

In [110]:
x_train_stem_rosrus.shape

(130000, 36766)

In [111]:
x_train_bow_svd_stem_rosrus.shape

(130000, 50)

### scaling

In [115]:
scaler_stem=StandardScaler()
scaler_stem_stopkept=StandardScaler()
scaler_lemma=StandardScaler()
scaler_lemma_stopkept=StandardScaler()

In [116]:
x_train_svd_stem_rosrus_scaled=scaler_stem.fit_transform(x_train_bow_svd_stem_rosrus)
x_test_svd_stem_rosrus_scaled=scaler_stem.transform(x_test_bow_svd_stem_rosrus)

In [117]:
x_train_svd_stem_stopkept_rosrus_scaled=scaler_stem_stopkept.fit_transform(x_train_bow_svd_stem_stopkept_rosrus)
x_test_svd_stem_stopkept_rosrus_scaled=scaler_stem_stopkept.transform(x_test_bow_svd_stem_stopkept_rosrus)

In [118]:
x_train_svd_lemma_rosrus_scaled=scaler_lemma.fit_transform(x_train_bow_svd_lemma_rosrus)
x_test_svd_lemma_rosrus_scaled=scaler_lemma.transform(x_test_bow_svd_lemma_rosrus)

In [119]:
x_train_svd_lemma_stopkept_rosrus_scaled=scaler_lemma_stopkept.fit_transform(x_train_bow_svd_lemma_stopkept_rosrus)
x_test_svd_lemma_stopkept_rosrus_scaled=scaler_lemma_stopkept.transform(x_test_bow_svd_lemma_stopkept_rosrus)

In [126]:
stem_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)
stem_stopkept_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)
lemma_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)
lemma_stopkept_svd_MLP=MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000,activation='relu',solver='adam',alpha=0.001)

### stem

In [127]:
stem_svd_MLP.fit(x_train_svd_stem_rosrus_scaled,y_train_bal)

y_pred_stem_mlp_train=stem_svd_MLP.predict(x_train_svd_stem_rosrus_scaled)
print(accuracy_score(y_pred_stem_mlp_train, y_train_bal))

y_pred_stem_mlp=stem_svd_MLP.predict(x_test_svd_stem_rosrus_scaled)
print(accuracy_score(y_pred_stem_mlp, y_test_bal))

0.6299076923076923
0.3688342679871137


In [130]:
results_Bow_rosrus['rosrus MLP stem+stopwords removed']=accuracy_score(y_pred_stem_mlp, y_test_bal)
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137}

### stem + stopwords kept

In [131]:
stem_stopkept_svd_MLP.fit(x_train_svd_stem_stopkept_rosrus_scaled,y_train_bal)

y_pred_stem_stopkept_mlp_train=stem_svd_MLP.predict(x_train_svd_stem_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_stem_stopkept_mlp_train, y_train_bal))

y_pred_stem_stopkept_mlp=stem_svd_MLP.predict(x_test_svd_stem_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_stem_stopkept_mlp, y_test_bal))

0.10434615384615385
0.07729387901205106


In [132]:
results_Bow_rosrus['rosrus MLP stem+stopwords kept']=accuracy_score(y_pred_stem_stopkept_mlp, y_test_bal)

### lemma

In [152]:
lemma_svd_MLP.fit(x_train_svd_lemma_rosrus_scaled,y_train_bal)

y_pred_lemma_mlp_train=stem_svd_MLP.predict(x_train_svd_lemma_rosrus_scaled)
print(accuracy_score(y_pred_lemma_mlp_train, y_train_bal))

y_pred_lemma_mlp=stem_svd_MLP.predict(x_test_svd_lemma_rosrus_scaled)
print(accuracy_score(y_pred_lemma_mlp, y_test_bal))

0.19458461538461538
0.12979358071829136


In [153]:
results_Bow_rosrus['rosrus MLP lemma+stopwords removed']=accuracy_score(y_pred_lemma_mlp, y_test_bal)

### lemma + stopwords kept

In [135]:
lemma_stopkept_svd_MLP.fit(x_train_svd_lemma_stopkept_rosrus_scaled,y_train_bal)

y_pred_lemma_stopkept_mlp_train=stem_svd_MLP.predict(x_train_svd_lemma_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_lemma_stopkept_mlp_train, y_train_bal))

y_pred_lemma_stopkept_mlp=stem_svd_MLP.predict(x_test_svd_lemma_stopkept_rosrus_scaled)
print(accuracy_score(y_pred_lemma_stopkept_mlp, y_test_bal))

0.11286923076923076
0.07932227657797399


In [136]:
results_Bow_rosrus['rosrus MLP lemma+stopwords kept']=accuracy_score(y_pred_lemma_stopkept_mlp, y_test_bal)

In [137]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137,
 'rosrus MLP stem+stopwords kept': 0.07729387901205106,
 'rosrus MLP lemma+stopwords removed': 0.12979358071829136,
 'rosrus MLP lemma+stopwords kept': 0.07932227657797399}

## Save results

In [160]:
joblib.dump(results_Bow_rosrus,'results_Bow_rosrus')

['results_Bow_rosrus']

In [161]:
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137,
 'rosrus MLP stem+stopwords kept': 0.07729387901205106,
 'rosrus MLP lemma+stopwords removed': 0.12979358071829136,
 'rosrus MLP lemma+stopwords kept': 0.07932227657797399,
 'rosrus SVC_BoW stem+StopWords removed': 0.46693711967545637,
 'rosrus MultinomialNB_BoW stem+StopWords removed': 0.5601479537048085,
 'rosrus SVC_BoW lemma+stopwords removed': 0.46404963608161315,
 'rosrus MultinomialNB_BoW lemma+stopwords removed': 0.5621763512

SVM performed best with stem + stopwords removed  --> 46.7%

NB performed best with lemma + stopwords removed  -->  56.22%

MLP performed best with stem + stopwords removed  --> 36.88%

HMM performed poorly due to the lack of sequential structure in short news headlines

In [242]:
results_Bow_rosrus=joblib.load('results_Bow_rosrus')
results_Bow_rosrus

{'rosrus SVC_BoW stem+StopWords kept': 0.45715308435747526,
 'rosrus MultinomialNB_BoW stem+StopWords kept': 0.559026369168357,
 'rosrus SVC_BoW Lemma+StopWords kept': 0.4530246987233027,
 'rosrus MultinomialNB_BoW Lemma+StopWords kept': 0.5595275026846438,
 'rosrus HMM_BoW stem+StopWords removed': 0.1699081255220141,
 'rosrus HMM_BoW stem+StopWords kept': 0.082877938193533,
 'rosrus HMM_BoW lemma+StopWords kept': 0.024245316787972794,
 'rosrus HMM_BoW lemma+StopWords removed': 0.060350793461400785,
 'rosrus MLP stem+stopwords removed': 0.3688342679871137,
 'rosrus MLP stem+stopwords kept': 0.07729387901205106,
 'rosrus MLP lemma+stopwords removed': 0.12979358071829136,
 'rosrus MLP lemma+stopwords kept': 0.07932227657797399,
 'rosrus SVC_BoW stem+StopWords removed': 0.46693711967545637,
 'rosrus MultinomialNB_BoW stem+StopWords removed': 0.5601479537048085,
 'rosrus SVC_BoW lemma+stopwords removed': 0.46404963608161315,
 'rosrus MultinomialNB_BoW lemma+stopwords removed': 0.5621763512

In [243]:
with open("results_Bow_rosrus.txt", "w", encoding="utf-8") as f:
    for key, value in results_Bow_rosrus.items():
        f.write(f"{key}: {value}\n")